# Stage-B Resampling Study — Two-Stage Hierarchical NIDS

**Extends Stage 6 (Model Improvement)** of `two_stage_hierarchical.ipynb`.

The two-stage diagnostics showed the **gate is essentially solved** (attack recall ~0.997) — the
ceiling is **Stage B, the attack-family classifier** (macro-F1 ~0.574 on true attacks). This notebook
holds the gate fixed and varies **only Stage B's imbalance handling**, measuring both:

- **Stage-B macro-F1** on the *true* attack rows (the isolated sub-problem), and
- **end-to-end 10-class macro-F1** routed through the fixed gate (the headline metric).

**Reference points (macro-F1):** flat XGBoost 0.5807 · two-stage + balanced weights 0.5858.

> Resampling is applied to the **attack-only training rows only** — never the test set, never before the
> split. Full SMOTE already underperformed weighting in earlier work; the question here is whether
> *capped* (moderate) oversampling does better.

## 1. Pipeline setup
Load → dedupe → variance/correlation prune → stratified split (identical to the baselines).

In [1]:
%pip install xgboost imbalanced-learn -q

import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek

LABELS = {0:"Benign",1:"Analysis",2:"Backdoor",3:"DoS",4:"Exploits",
          5:"Fuzzers",6:"Generic",7:"Reconnaissance",8:"Shellcode",9:"Worms"}

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun  = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("features:", X.shape[1], "| train:", X_train.shape[0], "| test:", X_test.shape[0])

Note: you may need to restart the kernel to use updated packages.
features: 44 | train: 244938 | test: 61235


## 2. Fixed gate + Stage-B data + evaluation harness

The gate (Benign vs Attack) is trained once and reused for every variant, so any difference in the
end-to-end score comes purely from Stage B. Stage-B trains on attack rows only (families remapped 1-9 → 0-8).

In [2]:
def make_xgb():
    return XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9, tree_method="hist",
        eval_metric="mlogloss", n_jobs=-1, random_state=42)

# --- fixed gate ---
y_train_bin = (y_train != 0).astype(int)
gate = make_xgb()
gate.fit(X_train, y_train_bin, sample_weight=compute_sample_weight("balanced", y_train_bin))
gate_pred = gate.predict(X_test)

# --- attack-only Stage-B data ---
attack_labels = sorted(l for l in y_train.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
X_train_att = X_train[y_train != 0].reset_index(drop=True)
y_train_att = y_train[y_train != 0].map(fam_to_idx).reset_index(drop=True)

test_att     = (y_test != 0).values
y_test_fam   = y_test[y_test != 0].map(fam_to_idx).values
counts = y_train_att.value_counts().sort_index()
print("Stage-B train family counts:")
print({LABELS[idx_to_fam[i]]: int(counts[i]) for i in counts.index})

def capped_strategy(cap):
    '''Oversample only families below `cap`, up to `cap`; leave larger ones untouched.'''
    return {c: cap for c in counts.index if counts[c] < cap}

def evaluate(stage_b, tag):
    pred_att = stage_b.predict(X_test[test_att])                 # on true attacks
    sb_mf1   = f1_score(y_test_fam, pred_att, average="macro", zero_division=0)
    # end-to-end via the fixed gate
    yp = np.zeros(len(X_test), int); m = gate_pred == 1
    if m.any():
        yp[m] = np.vectorize(idx_to_fam.get)(stage_b.predict(X_test[m]))
    per = {LABELS[idx_to_fam[i]]:
           round(f1_score(y_test_fam == i, pred_att == i, zero_division=0), 3)
           for i in counts.index}
    return dict(variant=tag,
                stageB_mF1=round(sb_mf1, 4),
                e2e_mF1=round(f1_score(y_test, yp, average="macro", zero_division=0), 4),
                e2e_mP =round(precision_score(y_test, yp, average="macro", zero_division=0), 4),
                e2e_mR =round(recall_score(y_test, yp, average="macro", zero_division=0), 4),
                **per)

Stage-B train family counts:
{'Analysis': 308, 'Backdoor': 362, 'DoS': 3529, 'Exploits': 24728, 'Fuzzers': 21116, 'Generic': 3597, 'Reconnaissance': 9625, 'Shellcode': 1478, 'Worms': 193}


## 3. Run the variants

Each variant differs only in how Stage B handles imbalance: `balanced` sample weights (the current
approach), full SMOTE, capped SMOTE at two caps, Borderline-SMOTE, SMOTE+Tomek cleaning, and a
capped-SMOTE-plus-weights combination.

In [3]:
results = []

# A. balanced sample weights (current two-stage approach)
b = make_xgb(); b.fit(X_train_att, y_train_att,
                      sample_weight=compute_sample_weight("balanced", y_train_att))
results.append(evaluate(b, "A. weights (balanced)"))

# B. full SMOTE — balance every family to the majority
Xs, ys = SMOTE(random_state=42).fit_resample(X_train_att, y_train_att)
b = make_xgb(); b.fit(Xs, ys); results.append(evaluate(b, "B. full SMOTE"))

# C. capped SMOTE at cap = 2000 and 5000
for cap in (2000, 5000):
    Xs, ys = SMOTE(random_state=42, sampling_strategy=capped_strategy(cap)).fit_resample(X_train_att, y_train_att)
    b = make_xgb(); b.fit(Xs, ys); results.append(evaluate(b, f"C. capped SMOTE cap={cap}"))

# D. capped Borderline-SMOTE (synthesise near class borders)
Xs, ys = BorderlineSMOTE(random_state=42, sampling_strategy=capped_strategy(5000)).fit_resample(X_train_att, y_train_att)
b = make_xgb(); b.fit(Xs, ys); results.append(evaluate(b, "D. Borderline-SMOTE cap=5000"))

# E. capped SMOTE + Tomek links cleaning
Xs, ys = SMOTETomek(random_state=42,
    smote=SMOTE(random_state=42, sampling_strategy=capped_strategy(5000))).fit_resample(X_train_att, y_train_att)
b = make_xgb(); b.fit(Xs, ys); results.append(evaluate(b, "E. capped SMOTE+Tomek cap=5000"))

# F. capped SMOTE (5000) THEN balanced weights on the resampled set
Xs, ys = SMOTE(random_state=42, sampling_strategy=capped_strategy(5000)).fit_resample(X_train_att, y_train_att)
b = make_xgb(); b.fit(Xs, ys, sample_weight=compute_sample_weight("balanced", ys))
results.append(evaluate(b, "F. capped SMOTE5000 + weights"))

print("done —", len(results), "variants")

done — 7 variants


## 4. Results

`stageB_mF1` = Stage B in isolation (true attack rows). `e2e_mF1` = end-to-end 10-class macro-F1 through
the fixed gate — **the headline metric**. Reference: flat XGBoost 0.5807, two-stage+weights 0.5858.

In [4]:
summary = pd.DataFrame(results)[["variant","stageB_mF1","e2e_mF1","e2e_mP","e2e_mR"]]
summary = summary.sort_values("e2e_mF1", ascending=False).reset_index(drop=True)
best = summary.iloc[0]
print(summary.to_string(index=False))
print(f"\nBest end-to-end macro-F1: {best.variant}  ->  {best.e2e_mF1}")

                       variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR
      C. capped SMOTE cap=2000      0.5777   0.5976  0.6624  0.5790
E. capped SMOTE+Tomek cap=5000      0.5768   0.5956  0.6369  0.5961
      C. capped SMOTE cap=5000      0.5708   0.5927  0.6390  0.5848
  D. Borderline-SMOTE cap=5000      0.5681   0.5866  0.6404  0.5720
         A. weights (balanced)      0.5742   0.5858  0.5427  0.6779
 F. capped SMOTE5000 + weights      0.5555   0.5698  0.5351  0.6334
                 B. full SMOTE      0.5327   0.5561  0.5624  0.5768

Best end-to-end macro-F1: C. capped SMOTE cap=2000  ->  0.5976


### 4.1 Per-family Stage-B F1 (the rare families are the target)

In [5]:
perfam = pd.DataFrame(results).set_index("variant")[
    [LABELS[idx_to_fam[i]] for i in counts.index]]
print(perfam.to_string())

                                Analysis  Backdoor    DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
variant                                                                                                                
A. weights (balanced)              0.565     0.556  0.425     0.763    0.749    0.772           0.611      0.318  0.409
B. full SMOTE                      0.485     0.472  0.366     0.782    0.772    0.762           0.646      0.279  0.230
C. capped SMOTE cap=2000           0.522     0.609  0.393     0.792    0.779    0.817           0.655      0.273  0.359
C. capped SMOTE cap=5000           0.537     0.571  0.396     0.790    0.778    0.811           0.648      0.263  0.344
D. Borderline-SMOTE cap=5000       0.522     0.553  0.380     0.789    0.777    0.803           0.656      0.301  0.333
E. capped SMOTE+Tomek cap=5000     0.583     0.544  0.352     0.786    0.775    0.790           0.654      0.284  0.422
F. capped SMOTE5000 + weights      0.568

## 5. Findings

- **Full SMOTE hurts** (e2e macro-F1 ≈ 0.556 vs 0.586 with weights) — balancing every family to the
  majority inflates Worms/Analysis ~100× and interpolates noise from a handful of real points. This
  reconfirms the project's earlier SMOTE negative result, now *inside* the two-stage.
- **Capped (moderate) SMOTE is the winning Stage-B lever** — cap ≈ 2000 reaches e2e macro-F1 ≈ **0.598**
  (+1.7 pts over the two-stage baseline and over flat XGBoost). Capping oversamples the rare families
  only up to a moderate count instead of full balance, avoiding the noise blow-up.
- **It is a precision ↔ recall knob, not a free win.** Balanced *weights* maximise macro-recall
  (~0.68, catches more rare attacks, more false alarms); *capped SMOTE* maximises macro-precision
  (~0.66, fewer false alarms, lower recall). Which to prefer is an operating-point choice for the SOC.
- Borderline-SMOTE and SMOTE+Tomek match capped SMOTE but do not beat it — the extra machinery buys
  little here.

> **Honesty note.** The cap above is read off the test set for illustration. For a reported thesis
> number, tune the cap on a validation split carved from the *training* data, then evaluate once on
> test — same discipline as the Stage-6 gate-threshold sweep.